# ST1504 Deep Learning — CA2 Part A
## Conditional DCGAN for CIFAR-10 Image Generation

**Team members:** `<Name 1, Admin No.>` & `<Name 2, Admin No.>`
**Module:** ST1504 Deep Learning · **Course:** DAAA · **AY2026/27 Semester 1**

---

### What this notebook covers

This notebook builds, trains, and evaluates a **Conditional Deep Convolutional GAN (cDCGAN)** that generates synthetic CIFAR-10 images across all 10 classes, as required by CA2 Part A (45 marks).

| Section | Maps to evaluation criterion |
|---|---|
| 1–3: Background, EDA, feature engineering | Background research, feature engineering (9 marks) |
| 4–7: Architecture, loss, training design | Application of GAN, explanation of architecture (9 marks) |
| 8–13: Smoke test, training, evaluation, improvement | Evaluation and improvement of GAN performance (9 marks) |
| 14: Discussion of brief's specific questions | Feeds directly into your write-up / presentation |
| 15: Export deliverables | Submission requirement (1000 images, best `.h5` weights) |
| Throughout | Quality of report — markdown explanation before every code block (9 marks) |

> ⚠️ **Honesty note on what was actually run.** This sample was prepared in a sandboxed environment **without internet access or a GPU**, so it cannot download CIFAR-10 or pretrained ImageNet weights. Cells that only need the *architecture* (model building, a synthetic-data "smoke test", the untrained class-conditional demo) **were genuinely executed** and show real output. Cells that need the **real dataset, a GPU, and/or internet** (EDA, the full 50-epoch training runs, Inception Score/FID) are fully implemented and ready to run, but their outputs are intentionally left blank here — **run them yourselves** (e.g. on Google Colab) so the losses, curves, images and scores in your submission are genuine results from your own training run, not numbers copied from a sample.

### Environment
- TensorFlow ≥ 2.15 (this notebook was developed and sanity-tested against **TF 2.21 / Keras 3**) — `pip install tensorflow` (GPU build) or `tensorflow-cpu`. A GPU runtime (e.g. Google Colab, T4/V100) is strongly recommended for the full training run.
- Standard scientific stack: `numpy`, `pandas`, `matplotlib`, `scipy`, `Pillow`.
- Internet access is required **once**, to download (a) CIFAR-10 via `tf.keras.datasets.cifar10.load_data()` and (b) pretrained ImageNet weights for `InceptionV3` if you run the optional Section 12 metrics.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from scipy.linalg import sqrtm

SEED = 1504
LATENT_DIM = 128
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 50          # full training run length (Section 9) -- reduce for quick local iteration
SAMPLE_EVERY = 10    # epochs between progress snapshots / checkpoints

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tf.keras.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

: 

*(Output above is from the CPU-only development sandbox used to verify this notebook — on your own GPU runtime, `GPU available` should print `True`, and you should confirm your TensorFlow/Keras version matches or is compatible with the API calls used below, e.g. `LeakyReLU(negative_slope=...)` vs. the older `LeakyReLU(alpha=...)` naming.)*

## 1. Background & Problem Statement

**Generative Adversarial Networks (GANs)** consist of two competing neural networks — a *generator* that learns to synthesise realistic data from random noise, and a *discriminator* that learns to distinguish real data from generated data. The two networks are trained jointly in a minimax game: the generator improves by trying to fool the discriminator, while the discriminator improves by getting better at spotting fakes (Goodfellow et al., 2014).

**Deep Convolutional GAN (DCGAN)** (Radford et al., 2015) adapted this idea to images by replacing fully-connected layers with strided/transposed convolutions, adding batch normalisation, and using ReLU/LeakyReLU activations — a set of architectural guidelines that made GAN training noticeably more stable for image generation, and which this notebook follows closely.

**Conditional GAN (cGAN)** (Mirza & Osindero, 2014) extends the vanilla GAN by feeding class-label information into both the generator and discriminator, so the generator can be *told* which class to produce and the discriminator can check that an image looks realistic **for its claimed class**. This is exactly what the assignment brief asks for: the ability to generate images "of a specific class" on demand, across all 10 CIFAR-10 classes.

**Why GAN over VAE for this task:** A Variational Autoencoder (VAE) optimises a reconstruction loss plus a KL-divergence regulariser, which tends to produce blurrier images because pixel-wise reconstruction losses average over plausible outputs. A GAN's adversarial loss instead pushes the generator toward the *mode* of realistic images, typically producing sharper textures — at the cost of trickier, less stable training (mode collapse, vanishing gradients). Given that CIFAR-10 images are already low-resolution (32×32) and rely heavily on texture/colour cues for recognisability, sharper GAN outputs are more likely to look like a specific class than a VAE's blurrier reconstructions. This trade-off (stability vs. sharpness) is revisited in Section 13.

**Problem statement:** Build a conditional DCGAN, trained only on the CIFAR-10 training set (no external data, no transfer learning, per the brief), that can generate 100 plausible images for each of the 10 classes (1000 images total), and evaluate the quality of the generated images both qualitatively and quantitatively.

## 2. Exploratory Data Analysis (EDA)

Before designing the network, we look at:
- **Class balance** — an imbalanced dataset would bias the discriminator/generator toward majority classes.
- **What the images actually look like** per class — informs the easier/harder-to-generate discussion in Section 14(c).
- **Pixel value range** — determines the normalisation strategy needed for the generator's output activation (Section 3).

> Requires internet access to download CIFAR-10 — not run in this sandbox. Run this yourself first.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print(f"x_train shape: {x_train.shape}, dtype: {x_train.dtype}")
print(f"y_train shape: {y_train.shape}, unique labels: {np.unique(y_train)}")
print(f"Pixel value range (raw): [{x_train.min()}, {x_train.max()}]")

unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CLASS_NAMES[u]:12s}: {c} images")

In [ ]:
fig, axes = plt.subplots(10, 8, figsize=(9, 12))
for class_id in range(10):
    class_images = x_train[y_train == class_id][:8]
    for i in range(8):
        ax = axes[class_id, i]
        ax.imshow(class_images[i])
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_ylabel(CLASS_NAMES[class_id], fontsize=9, rotation=0, labelpad=45, va='center')
plt.suptitle("CIFAR-10 sample images by class", y=1.0)
plt.tight_layout()
plt.savefig("eda_sample_grid.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar([CLASS_NAMES[u] for u in unique], counts, color='steelblue')
plt.title("Class distribution — CIFAR-10 training set")
plt.ylabel("Number of images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=100)
plt.show()

print("\nObservation: the training set is perfectly balanced (5000 images per class),")
print("so no class-weighting or resampling is needed before training the GAN.")

## 3. Feature Engineering & Data Pipeline

Three design decisions shape the input pipeline:

1. **Pixel normalisation to `[-1, 1]`.** The generator's final layer uses a `tanh` activation (standard DCGAN practice — stronger gradients than `sigmoid`, sharper outputs). `tanh` is bounded in `[-1, 1]`, so real images must be rescaled to the same range (`pixel/127.5 - 1.0`) for the discriminator to compare real and fake images on a like-for-like scale.

2. **Label representation via a learned `Embedding`, not one-hot.** A one-hot vector treats every class as equally (and maximally) dissimilar from every other class. An `Embedding` layer instead lets the network *learn* a dense vector per class during training — classes the network finds visually similar (e.g. `cat`/`dog`) can end up with more similar embeddings than dissimilar ones (e.g. `cat`/`ship`), giving the model more flexibility to share low-level features across related classes.

3. **Data augmentation — horizontal flip only.** We apply `tf.image.random_flip_left_right` to the real images seen by the discriminator, effectively doubling the diversity of real examples it sees and helping prevent it from overfitting to specific orientations faster than the generator can keep up (a common cause of early GAN instability). We deliberately **do not** use vertical flips or large rotations: vehicles (`automobile`, `truck`, `airplane`, `ship`) and animals in CIFAR-10 essentially never appear upside-down in practice, so training on vertically-flipped images would teach the discriminator to accept unrealistic orientations, which would leak into what the generator learns to produce.

No further feature engineering (e.g. cropping, colour-jitter) was applied — CIFAR-10 images are already small, centred, and reasonably uniform in scale, so additional engineering had a low expected pay-off relative to the added training complexity.

In [ ]:
def make_dataset(images, labels, batch_size=BATCH_SIZE, augment=True, shuffle=True, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=images.shape[0], seed=seed)

    def _prep(img, lbl):
        img = tf.cast(img, tf.float32)
        img = (img / 127.5) - 1.0
        if augment:
            img = tf.image.random_flip_left_right(img)
        return img, lbl

    ds = ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(x_train, y_train, augment=True, shuffle=True)
print(f"Number of training batches per epoch: {tf.data.experimental.cardinality(train_ds).numpy()}")

for imgs, lbls in train_ds.take(1):
    print(f"Batch image range after normalisation: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]")
    print(f"Batch shape: {imgs.shape}, label shape: {lbls.shape}")

## 4. Generator Architecture

The generator maps `(noise vector, class label)` → `32×32×3 image`:

| Stage | Operation | Output shape | Why |
|---|---|---|---|
| Input | noise `z ∈ R^128`, label `y` | (128,), (1,) | Latent size large enough to encode meaningful variation, small enough to train quickly on 32×32 images |
| Label path | `Embedding(10, 50)` → `Flatten` | (50,) | Learn a dense class representation (Section 3) |
| Fuse | `Concatenate([z, embedding])` | (178,) | Combine noise with class information *before* any spatial structure exists, so every later layer is class-aware |
| Project | `Dense(4·4·256)` → `BatchNorm` → `ReLU` → `Reshape(4,4,256)` | (4,4,256) | Turn the 1-D vector into a small spatial feature map to seed convolutional upsampling |
| Upsample ×3 | `Conv2DTranspose` (stride 2) → `BatchNorm` → `ReLU`, filters 256→128→64 | 4→8→16→32 | Progressively double spatial resolution while halving channel depth — the standard DCGAN generator template |
| Output | `Conv2DTranspose(3, stride 2, activation='tanh')` | (32,32,3) | RGB image in `[-1, 1]`, matching the normalised real images |

**Design rationale (DCGAN guidelines followed):**
- `BatchNormalization` after every layer except the output stabilises training by keeping activation statistics consistent — one of the main reasons DCGAN training is more stable than a plain fully-connected GAN.
- `ReLU` (not `LeakyReLU`) in the generator, following the original DCGAN paper's finding that it helps the model saturate faster and cover the output space more quickly.
- No fully-connected layers after the initial projection — everything else is convolutional, preserving spatial locality and keeping the parameter count manageable (~1.4M parameters).

In [ ]:
def build_generator():
    noise = tf.keras.Input(shape=(LATENT_DIM,), name="noise_input")
    label = tf.keras.Input(shape=(1,), dtype='int32', name="label_input")

    emb = tf.keras.layers.Embedding(NUM_CLASSES, 50, name="label_embedding")(label)
    emb = tf.keras.layers.Flatten()(emb)

    x = tf.keras.layers.Concatenate()([noise, emb])
    x = tf.keras.layers.Dense(4 * 4 * 256, use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Reshape((4, 4, 256))(x)

    for filters in [128, 64]:
        x = tf.keras.layers.Conv2DTranspose(filters, 4, strides=2, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)

    out = tf.keras.layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='tanh')(x)

    return tf.keras.Model([noise, label], out, name="generator")


generator = build_generator()
generator.summary()

## 5. Discriminator Architecture

The discriminator maps `(image, class label)` → `real/fake logit`, and is also class-conditional so it can penalise a *realistic-looking image of the wrong class*:

| Stage | Operation | Output shape | Why |
|---|---|---|---|
| Label path | `Embedding(10, 1024)` → `Reshape(32,32,1)` | (32,32,1) | Project the class label into a spatial map the same size as the image, so it can be concatenated as an extra "channel" |
| Fuse | `Concatenate([image, label_map])` | (32,32,4) | The discriminator now sees a 4-channel input (RGB + label channel) — it can only call an image "real" if it looks realistic **for that specific label channel** |
| Downsample ×3 | `Conv2D` (stride 2) → `LeakyReLU(0.2)` → `Dropout(0.3)`, filters 64→128→256 | 32→16→8→4 | Strided convolutions instead of pooling let the network learn its own downsampling — standard DCGAN discriminator design |
| Output | `Flatten` → `Dense(1)` | scalar logit | Raw logit (no sigmoid) — paired with `BinaryCrossentropy(from_logits=True)` for numerical stability |

**Design rationale:**
- `LeakyReLU(0.2)` (not `ReLU`) avoids dead neurons and keeps gradients flowing for negative activations — the DCGAN paper found this specifically important for the discriminator.
- `Dropout(0.3)` after every conv block is a deliberate deviation from vanilla DCGAN (which relies only on BatchNorm): with a *conditional* discriminator, dropout slows the discriminator down, since it effectively gets label information "for free" and can otherwise overpower the generator early in training.
- **No BatchNormalization** in the discriminator: applying BatchNorm to real and fake batches separately can leak batch statistics between the two, making it trivially easy for the discriminator to tell them apart from its own batch composition rather than image content — a well-known caveat in DCGAN-style training, so it is intentionally omitted here.

In [ ]:
def build_discriminator():
    img = tf.keras.Input(shape=(32, 32, 3), name="image_input")
    label = tf.keras.Input(shape=(1,), dtype='int32', name="label_input")

    emb = tf.keras.layers.Embedding(NUM_CLASSES, 32 * 32, name="label_embedding")(label)
    emb = tf.keras.layers.Reshape((32, 32, 1))(emb)

    x = tf.keras.layers.Concatenate()([img, emb])

    for filters in [64, 128, 256]:
        x = tf.keras.layers.Conv2D(filters, 4, strides=2, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(0.2)(x)
        x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Flatten()(x)
    out = tf.keras.layers.Dense(1)(x)

    return tf.keras.Model([img, label], out, name="discriminator")


discriminator = build_discriminator()
discriminator.summary()

## 6. Loss Function, Optimisers & Training Strategy

- **Loss:** `BinaryCrossentropy(from_logits=True)` for both networks, matching the discriminator's raw-logit output (Section 5).
- **One-sided label smoothing:** real-image targets are set to `0.9` instead of `1.0` when training the discriminator (fake targets stay at `0.0`). This is a well-known GAN stabilisation trick (Salimans et al., 2016) — it stops the discriminator from becoming overconfident, keeping gradients flowing to the generator for longer instead of saturating early.
- **Optimiser:** `Adam(learning_rate=2e-4, beta_1=0.5)` for both networks. The DCGAN paper specifically recommends `beta_1=0.5` instead of Adam's usual default `0.9` — lower momentum reduces oscillation during the adversarial min-max game.
- **A note on Keras' `training` argument:** when a Keras model is called directly (e.g. `discriminator([img, label])`) outside of `.fit()`, the `training` flag defaults to inference mode (`BatchNorm`/`Dropout` disabled) unless explicitly set. Our custom training loop (Section 7) is careful to pass `training=True` whenever a network's own weights are being updated by that step, and `training=False` when a network is only being used to produce inputs for the *other* network — a subtle, easy-to-miss bug in hand-written GAN training loops, called out explicitly here.

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
# Optimisers are created per-run in Sections 9 and 13, so baseline vs. improved
# runs can use different learning rates without redefining any training logic.

## 7. Training Step (`tf.function`)

`make_train_step` is a **factory function**: given a generator, discriminator, and their optimisers, it returns a compiled `tf.function` that performs one full discriminator+generator update. Wrapping it as a factory — rather than hard-coding global `generator`/`discriminator` variables — lets us reuse *exactly the same, already-tested* training logic later to fairly compare a "baseline" run against an "improved" run (Section 13): both runs call the identical function, only the models/optimisers passed in differ.

Each call performs two phases:
1. **Discriminator phase** — freeze the generator (`training=False`, so BatchNorm uses its running statistics rather than this batch's noisy statistics) and update only `discriminator.trainable_variables`, using real images (labelled `0.9`) and freshly generated fake images (labelled `0`).
2. **Generator phase** — draw *fresh* noise and labels (decorrelating this update from the discriminator's), run the generator in training mode, and update only `generator.trainable_variables` so the discriminator scores its output as real (label `1`).

In [ ]:
def make_train_step(generator, discriminator, g_opt, d_opt, loss_fn,
                     latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    @tf.function
    def train_step(real_images, real_labels):
        batch_size = tf.shape(real_images)[0]
        real_labels = tf.reshape(real_labels, (-1, 1))

        # --- Phase 1: Discriminator update ---
        noise = tf.random.normal((batch_size, latent_dim))
        fake_labels = tf.random.uniform((batch_size, 1), 0, num_classes, dtype=tf.int32)

        with tf.GradientTape() as tape:
            fake_images = generator([noise, fake_labels], training=False)
            real_pred = discriminator([real_images, real_labels], training=True)
            fake_pred = discriminator([fake_images, fake_labels], training=True)

            d_loss_real = loss_fn(tf.ones_like(real_pred) * 0.9, real_pred)  # one-sided label smoothing
            d_loss_fake = loss_fn(tf.zeros_like(fake_pred), fake_pred)
            d_loss = d_loss_real + d_loss_fake

        d_grads = tape.gradient(d_loss, discriminator.trainable_variables)
        d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))

        # --- Phase 2: Generator update (fresh noise/labels) ---
        noise = tf.random.normal((batch_size, latent_dim))
        fake_labels = tf.random.uniform((batch_size, 1), 0, num_classes, dtype=tf.int32)

        with tf.GradientTape() as tape:
            fake_images = generator([noise, fake_labels], training=True)
            fake_pred = discriminator([fake_images, fake_labels], training=True)
            g_loss = loss_fn(tf.ones_like(fake_pred), fake_pred)

        g_grads = tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))

        return d_loss, g_loss

    return train_step


def generate_sample_grid(generator, n_per_class=3, latent_dim=LATENT_DIM,
                          num_classes=NUM_CLASSES, class_names=CLASS_NAMES, seed=None):
    '''Generates an n_per_class x num_classes grid of images, one column per class.'''
    if seed is not None:
        tf.random.set_seed(seed)
    fig, axes = plt.subplots(n_per_class, num_classes, figsize=(num_classes * 1.3, n_per_class * 1.3))
    for c in range(num_classes):
        noise = tf.random.normal((n_per_class, latent_dim))
        labels = tf.fill((n_per_class, 1), c)
        imgs = generator([noise, labels], training=False).numpy()
        imgs = ((imgs + 1) * 127.5).astype(np.uint8)
        for r in range(n_per_class):
            ax = axes[r, c] if n_per_class > 1 else axes[c]
            ax.imshow(imgs[r])
            ax.axis("off")
            if r == 0:
                ax.set_title(class_names[c], fontsize=8)
    plt.tight_layout()
    return fig

## 8. Sanity Check — Smoke Test on Synthetic Data

Before committing to a full 50-epoch training run, we run `train_step` for a few iterations on **randomly generated dummy data** with the same shapes as real CIFAR-10 batches. This is a cheap way to catch shape mismatches, `NaN`/`Inf` losses, or gradient bugs (e.g. the `training=True/False` mistake described in Section 6) *before* spending GPU time on the real dataset.

**This cell was genuinely executed during development** (output below is real, captured directly from running the exact code in this cell) — it confirms both loss values are finite and change step to step, i.e. gradients are flowing correctly through both networks, and that `save_weights(..., ".weights.h5")` works with the TensorFlow/Keras version used.

In [ ]:
print("Running smoke test with synthetic data (no dataset download required)...")

dummy_images = tf.random.uniform((8, 32, 32, 3), -1, 1)
dummy_labels = tf.random.uniform((8,), 0, NUM_CLASSES, dtype=tf.int32)
dummy_ds = tf.data.Dataset.from_tensor_slices((dummy_images, dummy_labels)).batch(4)

test_gen = build_generator()
test_disc = build_discriminator()
test_g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
test_d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
test_step = make_train_step(test_gen, test_disc, test_g_opt, test_d_opt, loss_fn)

for step, (imgs, lbls) in enumerate(dummy_ds.repeat(3)):
    d_l, g_l = test_step(imgs, lbls)
    assert np.isfinite(float(d_l)) and np.isfinite(float(g_l)), "Non-finite loss detected!"
    print(f"  step {step}: D_loss={float(d_l):.4f}  G_loss={float(g_l):.4f}")

print("\nSmoke test passed: shapes, gradients and losses are all correct.")

test_gen.save_weights("smoke_test_generator.weights.h5")
print("save_weights(...'.weights.h5') succeeded.")

## 9. Baseline Model — Full Training Run

We now instantiate a fresh generator/discriminator pair and train for `EPOCHS` epochs on the **real, normalised, augmented CIFAR-10 training set** built in Section 3.

> ⚠️ **Run this on a GPU runtime.** On a single T4/V100 GPU, one epoch over all 50,000 training images at `batch_size=128` typically takes on the order of tens of seconds to ~1 minute; the full 50-epoch baseline run should comfortably fit a typical lab/Colab session. On CPU-only hardware this can take 10–20× longer — reduce `EPOCHS` for quick iteration, then increase it for your final submission run.
>
> Every `SAMPLE_EVERY` epochs we (a) print the epoch-averaged losses, (b) save a visual grid of generated images per class so you can *see* training progress and catch mode collapse early, and (c) checkpoint the generator's weights.
>
> **Not executed in this sandbox** (needs the real dataset + GPU time) — run it yourself.

In [ ]:
def train_gan(generator, discriminator, dataset, epochs, g_opt, d_opt, loss_fn,
              run_name="baseline", sample_every=SAMPLE_EVERY, checkpoint_dir="checkpoints",
              latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    os.makedirs(checkpoint_dir, exist_ok=True)
    train_step = make_train_step(generator, discriminator, g_opt, d_opt, loss_fn, latent_dim, num_classes)
    history = {"d_loss": [], "g_loss": []}

    for epoch in range(1, epochs + 1):
        epoch_d, epoch_g = [], []
        t0 = time.time()
        for real_images, real_labels in dataset:
            d_loss, g_loss = train_step(real_images, real_labels)
            epoch_d.append(float(d_loss))
            epoch_g.append(float(g_loss))

        avg_d, avg_g = float(np.mean(epoch_d)), float(np.mean(epoch_g))
        history["d_loss"].append(avg_d)
        history["g_loss"].append(avg_g)
        print(f"[{run_name}] epoch {epoch:3d}/{epochs} | D={avg_d:.4f} | G={avg_g:.4f} | {time.time()-t0:.1f}s")

        if epoch % sample_every == 0 or epoch == epochs:
            fig = generate_sample_grid(generator, n_per_class=3)
            fig.suptitle(f"{run_name} — epoch {epoch}")
            fig.savefig(f"{checkpoint_dir}/{run_name}_epoch{epoch:03d}.png", dpi=100, bbox_inches="tight")
            plt.show()
            plt.close(fig)
            generator.save_weights(f"{checkpoint_dir}/{run_name}_generator_epoch{epoch:03d}.weights.h5")

    return history


# --- Baseline run ---
tf.keras.utils.set_random_seed(SEED)
baseline_generator = build_generator()
baseline_discriminator = build_discriminator()
baseline_g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
baseline_d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

baseline_history = train_gan(
    baseline_generator, baseline_discriminator, train_ds,
    epochs=EPOCHS, g_opt=baseline_g_opt, d_opt=baseline_d_opt, loss_fn=loss_fn,
    run_name="baseline", sample_every=SAMPLE_EVERY,
)

## 10. Training Curves

For GANs, loss curves don't behave like a supervised model's (steadily decreasing) loss — because the two networks are adversaries, a "good" run typically shows both losses **oscillating and roughly stabilising** rather than converging to zero. If `D_loss` collapses toward 0 while `G_loss` grows without bound, the discriminator has "won" and stopped providing a useful learning signal (or vice-versa) — a sign of an imbalanced run worth revisiting via the learning-rate experiment in Section 13.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(baseline_history["d_loss"], label="Discriminator loss")
plt.plot(baseline_history["g_loss"], label="Generator loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Baseline cDCGAN training curves")
plt.legend(); plt.tight_layout()
plt.savefig("baseline_training_curves.png", dpi=100)
plt.show()

## 11. Qualitative Evaluation — "Eye-Power" Method

Per the brief, we generate a batch of images and manually classify each as **Clear** (recognisable, plausible instance of the class), **Marginal** (some correct features but noticeably flawed/blurry), or **Nonsense** (no recognisable structure). We generate 10 images per class (100 total) for review.

Fill in the `clear` / `marginal` / `nonsense` counts in the `scores` dictionary below **after visually inspecting the grid**, then re-run the summary cell to get a per-class and overall quality percentage for your report.

In [ ]:
eval_fig = generate_sample_grid(baseline_generator, n_per_class=10, seed=SEED)
eval_fig.suptitle("Baseline model — 10 samples per class for by-eye evaluation", y=1.02)
eval_fig.savefig("eyepower_evaluation_grid.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Fill in these counts (should sum to 10 per class) after visually reviewing the grid above.
scores = {
    "airplane":   {"clear": None, "marginal": None, "nonsense": None},
    "automobile": {"clear": None, "marginal": None, "nonsense": None},
    "bird":       {"clear": None, "marginal": None, "nonsense": None},
    "cat":        {"clear": None, "marginal": None, "nonsense": None},
    "deer":       {"clear": None, "marginal": None, "nonsense": None},
    "dog":        {"clear": None, "marginal": None, "nonsense": None},
    "frog":       {"clear": None, "marginal": None, "nonsense": None},
    "horse":      {"clear": None, "marginal": None, "nonsense": None},
    "ship":       {"clear": None, "marginal": None, "nonsense": None},
    "truck":      {"clear": None, "marginal": None, "nonsense": None},
}

scores_df = pd.DataFrame(scores).T
scores_df["total"] = scores_df.sum(axis=1)
scores_df["% clear"] = 100 * scores_df["clear"] / scores_df["total"]
scores_df.sort_values("% clear", ascending=False)

## 12. Quantitative Evaluation — Inception Score & Fréchet Inception Distance (Optional)

The brief allows an optional quantitative metric alongside by-eye scoring. We implement both **Inception Score (IS)** (Salimans et al., 2016) and **Fréchet Inception Distance (FID)** (Heusel et al., 2017), using a pretrained `InceptionV3` (ImageNet weights) as a fixed feature extractor:

- **Inception Score** rewards images that are (a) confidently classified into a single class by Inception (sharp, recognisable) and (b) collectively diverse across many classes (not mode-collapsed). Higher is better.
- **FID** compares the distribution of Inception feature-space activations between real and generated images (mean + covariance), rather than relying on classifier confidence. Lower is better — 0 would mean the two distributions are identical.

**Important caveat:** `InceptionV3` was trained on 1000 *ImageNet* classes, not the 10 CIFAR-10 classes, and expects 299×299 inputs (our images are upsampled from 32×32, which adds no real detail). The absolute numbers are therefore **not directly comparable to published CIFAR-10 GAN benchmarks** — treat IS/FID here as a *relative* measure for comparing our own baseline vs. improved runs (Section 13), not an absolute quality claim.

`calculate_inception_score` and `calculate_fid` below were unit-tested independently on synthetic feature/probability arrays during development (confirming IS is high for confident+diverse predictions and near 1 for uniform noise, and FID is ≈0 for near-identical distributions and large for clearly shifted ones) — only the `InceptionV3` download/inference itself needs internet + GPU, so set `RUN_QUANT_EVAL = True` below once that's available.

In [ ]:
RUN_QUANT_EVAL = False  # set True when running with internet access + GPU/time available


def get_inception_models():
    is_model = tf.keras.applications.InceptionV3(include_top=True, weights='imagenet')
    fid_model = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet', pooling='avg')
    return is_model, fid_model


def preprocess_for_inception(images_uint8):
    '''images_uint8: (N,32,32,3) in [0,255] -> (N,299,299,3) preprocessed for InceptionV3.'''
    imgs = tf.image.resize(images_uint8, (299, 299), method='bilinear')
    imgs = tf.keras.applications.inception_v3.preprocess_input(imgs)
    return imgs


def calculate_inception_score(preds, n_split=10, eps=1e-16):
    '''preds: (N, num_imagenet_classes) softmax probabilities.'''
    n = preds.shape[0]
    idx = np.arange(n)
    np.random.shuffle(idx)
    splits = np.array_split(idx, n_split)
    scores = []
    for split_idx in splits:
        part = preds[split_idx]
        py = np.mean(part, axis=0, keepdims=True)
        kl = np.sum(part * (np.log(part + eps) - np.log(py + eps)), axis=1)
        scores.append(np.exp(np.mean(kl)))
    return float(np.mean(scores)), float(np.std(scores))


def calculate_fid(feat_real, feat_fake):
    mu1, sigma1 = feat_real.mean(axis=0), np.cov(feat_real, rowvar=False)
    mu2, sigma2 = feat_fake.mean(axis=0), np.cov(feat_fake, rowvar=False)
    diff = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2.0 * covmean))


def evaluate_quantitative(generator, real_images_uint8, n_samples=1000, batch_size=50):
    is_model, fid_model = get_inception_models()

    fake_imgs = []
    per_class = n_samples // NUM_CLASSES
    for c in range(NUM_CLASSES):
        noise = tf.random.normal((per_class, LATENT_DIM))
        labels = tf.fill((per_class, 1), c)
        imgs = generator([noise, labels], training=False).numpy()
        fake_imgs.append(((imgs + 1) * 127.5).astype(np.uint8))
    fake_imgs = np.concatenate(fake_imgs, axis=0)

    real_sample = real_images_uint8[np.random.choice(len(real_images_uint8), n_samples, replace=False)]

    def batched_predict(model, images):
        outs = []
        for i in range(0, len(images), batch_size):
            batch = preprocess_for_inception(images[i:i + batch_size])
            outs.append(model.predict(batch, verbose=0))
        return np.concatenate(outs, axis=0)

    fake_is_preds = batched_predict(is_model, fake_imgs)
    is_mean, is_std = calculate_inception_score(fake_is_preds)

    real_feats = batched_predict(fid_model, real_sample)
    fake_feats = batched_predict(fid_model, fake_imgs)
    fid = calculate_fid(real_feats, fake_feats)

    return {"inception_score_mean": is_mean, "inception_score_std": is_std, "fid": fid}


if RUN_QUANT_EVAL:
    baseline_metrics = evaluate_quantitative(baseline_generator, x_train)
    print(f"Baseline — Inception Score: {baseline_metrics['inception_score_mean']:.3f} "
          f"± {baseline_metrics['inception_score_std']:.3f} | FID: {baseline_metrics['fid']:.2f}")
else:
    print("RUN_QUANT_EVAL is False — skipping. Set to True with internet/GPU access to compute real IS/FID scores.")

## 13. Systematic Improvement — Two-Timescale Update Rule (TTUR)

**Diagnosis:** with equal learning rates (Section 6), the discriminator's task (binary real/fake classification, made easier by the conditional label channel) is fundamentally simpler than the generator's task (produce a full, class-consistent 32×32×3 image). If the discriminator gets too far ahead of the generator, the gradients it passes back become uninformative — a common cause of stalled GAN training.

**Fix — TTUR (Heusel et al., 2017):** use *different* learning rates for the two networks, so the discriminator updates more conservatively while the generator is given comparatively more room to move. We compare this against the baseline using **identical architecture, data pipeline and training procedure** (the same `make_train_step`/`train_gan` functions from Sections 7–9), changing only the two learning rates — this isolates the effect of the one variable being tested, which is exactly the systematic, one-change-at-a-time approach the brief asks for.

Other improvement directions considered but left for future work due to time constraints (worth mentioning in your report/presentation to show breadth of understanding):
- **Spectral normalisation** on discriminator conv layers, to enforce a Lipschitz constraint and further stabilise training (SN-GAN, Miyato et al., 2018).
- **Self-attention layers** in the generator, to better capture long-range spatial structure than plain convolutions (SAGAN/BigGAN-style architectures) — likely to matter more at higher resolutions than CIFAR-10's 32×32.
- **Best-checkpoint selection via periodic FID** rather than saving on a fixed schedule, so the final exported weights correspond to the epoch with the best measured quality rather than simply the last epoch trained.

> **Not executed in this sandbox** — run this after Section 9, on the same GPU runtime.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
improved_generator = build_generator()
improved_discriminator = build_discriminator()
improved_g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)   # generator keeps the baseline LR
improved_d_opt = tf.keras.optimizers.Adam(5e-5, beta_1=0.5)   # discriminator LR lowered 4x (TTUR)

improved_history = train_gan(
    improved_generator, improved_discriminator, train_ds,
    epochs=EPOCHS, g_opt=improved_g_opt, d_opt=improved_d_opt, loss_fn=loss_fn,
    run_name="improved_ttur", sample_every=SAMPLE_EVERY,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].plot(baseline_history["d_loss"], label="D"); axes[0].plot(baseline_history["g_loss"], label="G")
axes[0].set_title("Baseline (equal LR)"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(improved_history["d_loss"], label="D"); axes[1].plot(improved_history["g_loss"], label="G")
axes[1].set_title("Improved (TTUR)"); axes[1].legend(); axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig("baseline_vs_improved_curves.png", dpi=100)
plt.show()

if RUN_QUANT_EVAL:
    improved_metrics = evaluate_quantitative(improved_generator, x_train)
    comparison = pd.DataFrame([
        {"run": "baseline", **baseline_metrics},
        {"run": "improved_ttur", **improved_metrics},
    ])
    display(comparison)
else:
    print("Set RUN_QUANT_EVAL = True in Section 12 to also compare IS/FID numerically.")

## 14. Discussion — Answering the Brief's Specific Questions

### (a) "If you are asked to generate images of a specific class, propose a way of doing it."

This is exactly what conditioning solves (Sections 4–5): because both networks receive the class label as an explicit input, generating a specific class is as simple as fixing the `label` input to the generator while sampling random noise (see `generate_class` below). No retraining and no separate model per class is needed — a single conditional model covers all 10 classes, far more parameter-efficient than training 10 separate unconditional GANs.

In [ ]:
def generate_class(generator, class_id, n=10, latent_dim=LATENT_DIM):
    noise = tf.random.normal((n, latent_dim))
    labels = tf.fill((n, 1), class_id)
    imgs = generator([noise, labels], training=False).numpy()
    return ((imgs + 1) * 127.5).astype(np.uint8)


# Demo on the UNTRAINED generator from Section 4 -- purely to show the mechanism
# and confirm the output shape/range are correct. After running Section 9's
# training, call this on `baseline_generator` to get actual recognisable cats.
cat_images = generate_class(generator, class_id=CLASS_NAMES.index("cat"), n=10)
print("cat_images shape:", cat_images.shape, " dtype:", cat_images.dtype,
      " range:", cat_images.min(), "-", cat_images.max())

fig, axes = plt.subplots(1, 10, figsize=(14, 1.6))
for i in range(10):
    axes[i].imshow(cat_images[i])
    axes[i].axis("off")
fig.suptitle("generate_class(generator, class_id='cat') -- UNTRAINED weights, mechanism/shape demo only", fontsize=9)
plt.tight_layout()
plt.savefig("cat_demo.png", dpi=100, bbox_inches="tight")
plt.show()

### (b) "If you are asked to generate black-and-white images instead of coloured ones, do you think it would be easier or harder?"

**Likely easier overall, but with a trade-off:**
- **Smaller output space.** A grayscale image has 1 channel instead of 3, so the generator's output layer and the discriminator's input have roughly a third of the values to get right — fewer degrees of freedom for the generator to get wrong, and fewer combinations for the discriminator to check.
- **Colour is often ambiguous for CIFAR-10 anyway** — e.g. `automobile`/`truck` appear in many real-world colours, so colour is not a reliable class signal for those classes, and the generator "wastes" capacity modelling colour variation that doesn't actually help class-consistency.
- **But** colour is a genuinely useful cue for *some* classes (e.g. `frog` green, `deer` brown, `ship`/`airplane` sky/water backgrounds) — removing it could make some class boundaries harder for the discriminator (and for a human doing the by-eye test) to tell apart, even though the pixel-level generation task is simpler.

To test this without a full retrain, only the generator's output layer and the discriminator's input concatenation need to change (shown below as an architecture proof-of-concept — deliberately **not trained further here**, since the brief only asks to "propose a way of doing it").

In [ ]:
def build_generator_grayscale():
    '''Same as build_generator(), but the final layer outputs 1 channel instead of 3.'''
    noise = tf.keras.Input(shape=(LATENT_DIM,))
    label = tf.keras.Input(shape=(1,), dtype='int32')
    emb = tf.keras.layers.Flatten()(tf.keras.layers.Embedding(NUM_CLASSES, 50)(label))
    x = tf.keras.layers.Concatenate()([noise, emb])
    x = tf.keras.layers.Dense(4 * 4 * 256, use_bias=False)(x)
    x = tf.keras.layers.ReLU()(tf.keras.layers.BatchNormalization()(x))
    x = tf.keras.layers.Reshape((4, 4, 256))(x)
    for f in [128, 64]:
        x = tf.keras.layers.Conv2DTranspose(f, 4, 2, 'same', use_bias=False)(x)
        x = tf.keras.layers.ReLU()(tf.keras.layers.BatchNormalization()(x))
    out = tf.keras.layers.Conv2DTranspose(1, 4, 2, 'same', activation='tanh')(x)  # <-- 1 channel, not 3
    return tf.keras.Model([noise, label], out, name="generator_grayscale")


gray_gen = build_generator_grayscale()
gray_gen.summary()

# shape check
noise = tf.random.normal((4, LATENT_DIM))
labels = tf.fill((4, 1), 3)
out = gray_gen([noise, labels], training=False)
print("grayscale output shape:", out.shape, " (expected (4, 32, 32, 1))")

# Real images would need tf.image.rgb_to_grayscale(images) added to the
# preprocessing pipeline in Section 3 to match this generator's output.

### (c) "What class(es) is/are relatively easier/harder to generate? Why?"

Base this section on **your own** results from Section 11 (by-eye `% clear`) and/or Section 12 (FID-per-class, if computed). As a general expectation to compare your results against — commonly seen across CIFAR-10 GAN literature and consistent with the structure of this dataset:

- **Typically easier: `airplane`, `automobile`, `ship`, `truck`, `horse`.** These classes have relatively rigid, consistent geometric shapes and more predictable backgrounds (sky, road, water) across training images, giving the generator a more consistent visual pattern to learn.
- **Typically harder: `cat`, `dog`, `bird`, `deer`, `frog`.** Animal classes have far more intra-class variation — pose, camera angle, background clutter, fine-grained texture (fur, feathers) — which is genuinely difficult for a relatively small DCGAN (a few million parameters) to capture at 32×32 resolution. Fine textures like fur are also disproportionately punished by the human eye during by-eye scoring, even when the overall shape is roughly right.

**Fill in your own per-class `% clear` numbers from Section 11 here, and discuss whether they match this expectation** — a mismatch is just as interesting to discuss as a match (e.g. your model's `frog` samples might come out surprisingly clear because frogs are usually photographed against a strongly contrasting green background).

## 15. Export Final Deliverables — 1000 Generated Images & Best Weights

Per the submission requirements, we now export:
1. **1000 generated images** — 100 per class × 10 classes — saved as individual `.png` files.
2. **Best generator weights** (`.h5`) — for reproducibility without retraining.

> Run this **after** training (Section 9, and optionally Section 13) completes and you have chosen which run produced the best results by your evaluation in Sections 11–13.

In [ ]:
BEST_GENERATOR = baseline_generator  # <-- change to improved_generator (or a loaded checkpoint) if it scored better

out_dir = Path("generated_images")
out_dir.mkdir(exist_ok=True)

IMAGES_PER_CLASS = 100
for class_id in range(NUM_CLASSES):
    noise = tf.random.normal((IMAGES_PER_CLASS, LATENT_DIM))
    labels = tf.fill((IMAGES_PER_CLASS, 1), class_id)
    imgs = BEST_GENERATOR([noise, labels], training=False).numpy()
    imgs = ((imgs + 1) * 127.5).astype(np.uint8)
    for i in range(IMAGES_PER_CLASS):
        Image.fromarray(imgs[i]).save(out_dir / f"{CLASS_NAMES[class_id]}_{i:03d}.png")

n_saved = len(list(out_dir.glob("*.png")))
print(f"Saved {n_saved} images to {out_dir}/ (expected {NUM_CLASSES * IMAGES_PER_CLASS})")

BEST_GENERATOR.save_weights("best_generator.weights.h5")
print("Saved best_generator.weights.h5")

## 16. Conclusion

*(Fill in after running the full training + evaluation above.)* Summarise: which run (baseline vs. TTUR-improved) performed better and by which measure (by-eye %, FID, IS, or visual inspection of mode collapse in the sample grids); which classes were easiest/hardest and whether that matched the expectation in Section 14(c); and what you would try next given more time (e.g. the spectral-normalisation / self-attention / longer-training ideas raised in Section 13).

## 17. References

- Goodfellow, I., Pouget-Abadie, J., Mirza, M., Xu, B., Warde-Farley, D., Ozair, S., Courville, A., & Bengio, Y. (2014). Generative Adversarial Networks. *NeurIPS*.
- Radford, A., Metz, L., & Chintala, S. (2015). Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks (DCGAN). *arXiv:1511.06434*.
- Mirza, M., & Osindero, S. (2014). Conditional Generative Adversarial Nets. *arXiv:1411.1784*.
- Salimans, T., Goodfellow, I., Zaremba, W., Cheung, V., Radford, A., & Chen, X. (2016). Improved Techniques for Training GANs. *NeurIPS*.
- Heusel, M., Ramsauer, H., Unterthiner, T., Nessler, B., & Hochreiter, S. (2017). GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium (FID/TTUR). *NeurIPS*.
- Miyato, T., Kataoka, T., Koyama, M., & Yoshida, Y. (2018). Spectral Normalization for Generative Adversarial Networks. *ICLR*.
- Krizhevsky, A. (2009). Learning Multiple Layers of Features from Tiny Images (CIFAR-10 dataset). University of Toronto.

*(Add any further sources you cite in your own background research, per SP's citation guide: https://sp-sg.libguides.com/citation)*

## 18. Group Contribution Statement

**Note:** per the brief, the contribution statement must be submitted as a **separate `.docx` file**, not inside this notebook — listing each member's specific contributions and % workload across Part A's deliverables (notebook, `.html` export, `.h5` weights, slides, 1000 images).